# Oxford Tutorial · Day 5: 规模实验与营销因果

## Persona (Oxford Tutorial Fellow)

You are an Oxford tutorial fellow in **规模实验与营销因果** (Scale Experiments & Marketing Causal Inference).

**Tutorial rules (strict):**
- Never give direct answers. 不直接给答案, 不直接答, 禁直接答案.
- Use Socratic questioning: every turn ends with a probing question.
- Reject vague claims: 当学生说"MAB 更好""CATE 有用", 追问"凭什么? 依据是什么? 给反例."
- Play devil's advocate: 当学生给出结论, 你要构造反例或反事实场景挑战.
- Scaffold fade: 学生连续 2 次答不出, 降一级脚手架 (从追问 -> 给 hint -> 给 worked example 指引), 但仍禁直接答案.
- 限频: 每单元每天 1 次 tutorial (防依赖, 参考 Oxford 每周 1 次 + Vygotsky 共构).

**Topic scope**: Thompson Sampling MAB on NSW real response rates / CausalForestDML CATE / Uplift Modeling 四类用户 / 营销因果三陷阱.

## Pre-Tutorial Task (强制 Retrieval Practice)

**Before entering the tutorial, you MUST submit (in writing, <=300 words):**

1. 在 NSW 数据上, Thompson Sampling MAB 相比固定 A/B 节省了多少实验成本? 给出具体数字与计算依据.
2. CATE 估计告诉你哪类用户响应最大? 写出 top-3 调节变量并解释因果机制.
3. Uplift Modeling 把用户分四类, 你会向哪一类投放优惠券? 为什么不投其他三类?

> 牛津 tutorial 原则: 没有预先准备 = 没有学习. Tutorial 时间用来辩护与挑战, 不用来讲授. 若 pre-task 未提交, tutorial 直接取消, 计 0 分.

**Pre-task 提交后, tutorial.ipynb 才会启动 cell 3 的 Socratic loop.**

In [ ]:
# Socratic Tutorial Loop (静态 if/else 模拟 4 轮追问, 不调 API)
# 每轮检测 defense (学生辩护) 是否含关键词, 失败则降一级 scaffold

rounds = [
    {
        "round": 1,
        "topic": "Thompson MAB 探索-利用权衡",
        "student_defense": "Thompson 比 epsilon-greedy 好, 因为后验采样自动平衡.",
        "challenge": "凭什么说'自动平衡'? 后验采样如何具体实现平衡? 给一个反例: 若某臂从未被拉过, Beta(1,1) 采样会发生什么? 这与 epsilon-greedy 的纯随机探索有何本质区别?",
        "socratic_q": "为什么 Beta(1,1) 的先验在数学上等价于'对未知臂保持乐观'? 这与 UCB 的置信上界有何异同?"
    },
    {
        "round": 2,
        "topic": "NSW 响应率驱动 MAB 的实验成本",
        "student_defense": "MAB 在 NSW 上节省了 8% 实验成本.",
        "challenge": "8% 这个数字依据是什么? 如何定义'实验成本'? 若把对照组响应率 0.35 作为 baseline, MAB 累计转化比固定 A/B 高多少? 反例: 若 NSW 响应率差距极小 (0.35 vs 0.36), MAB 还有优势吗?",
        "socratic_q": "假设 NSW 响应率差距从 0.07 变为 0.01, Thompson MAB 的 regret 曲线会如何变化? 为什么?"
    },
    {
        "round": 3,
        "topic": "CATE 与安慰剂检验",
        "student_defense": "CATE 显示年龄大的用户响应更大.",
        "challenge": "凭什么说年龄是调节变量? CausalForestDML 的 feature_importance 排第几? 反例: 若安慰剂检验 (随机分配 treat) 也显示年龄调节, 你的结论还成立吗? 如何排除混杂?",
        "socratic_q": "如何用反事实解释'为何年龄大的用户响应更大'? 是因为培训内容匹配度, 还是历史收入差异 (re75)?"
    },
    {
        "round": 4,
        "topic": "Uplift 四类用户与精准投放",
        "student_defense": "应该向 persuadables 投放优惠券.",
        "challenge": "如何识别 persuadables? 用 CATE 阈值还是 Qini 拐点? 反例: 若你的模型把 sleeping dogs 误判为 persuadables, 投放后会怎样? 营销因果三陷阱中, 哪个最容易在你这个投放策略中出现?",
        "socratic_q": "假设把投放预算从 30% 变为 60% (按 Qini 拐点之外继续投), 增量转化会如何变化? 为什么 Qini 拐点之外是浪费?"
    }
]

for r in rounds:
    print(f"\n=== Round {r['round']}: {r['topic']} ===")
    print(f"[Student]: {r['student_defense']}")
    print(f"[Fellow (devil's advocate)]: {r['challenge']}")
    print(f"[Socratic 追问]: {r['socratic_q']}")
    print("--- 学生需在下一轮回答 Socratic 追问, Fellow 检测是否含关键词 ---")
    # 静态 if/else 检测 defense 失败 -> 降一级 scaffold
    defense_keywords = {"后验", "Beta", "regret", "feature_importance", "安慰剂", "Qini", "persuadables"}
    if not any(k in r['student_defense'] for k in defense_keywords):
        print(f"[Scaffold 降级]: defense 缺关键词 -> 给 hint: 参考 practice.md D{r['round']} worked example")
    else:
        print(f"[Scaffold 保持]: defense 含关键词 -> 继续追问")


In [ ]:
# Student Model: 记录掌握度/盲点, 跨单元复用 (student_model.json)
import json, os

student_model = {
    "unit": "U-skill3-day5",
    "mastery": {
        "D1_Thompson_MAB": 0.6,
        "D2_CATE_CausalForestDML": 0.4,
        "D3_Uplift_Qini": 0.3
    },
    "blind_spots": [
        "Beta 后验采样的数学推导",
        "安慰剂检验的因果含义",
        "Qini 拐点与预算优化的关系"
    ],
    "weak_drill": "D2",
    "tutorial_history": [
        {"date": "2026-07-25", "rounds_completed": 4, "defense_keywords_hit": ["Beta", "regret"]}
    ],
    "next_review": {
        "C1_Thompson": 1,
        "C2_CATE": 1,
        "C3_Uplift": 1,
        "C4_三陷阱": 1
    },
    "cross_unit_links": [
        "skill-3-causal/day-2-ab-testing (A/B 基础)",
        "skill-3-causal/day-4-causal-ml (因果森林基础)",
        "skill-5-agentic/day-X (Agent 评估中的因果)"
    ]
}

model_path = "student_model.json"
if os.path.exists(model_path):
    with open(model_path, "r", encoding="utf-8") as f:
        old = json.load(f)
    old["tutorial_history"].extend(student_model["tutorial_history"])
    old["mastery"].update(student_model["mastery"])
    student_model = old

with open(model_path, "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)

print(f"Student model written to {model_path}")
print(f"Current mastery: {student_model['mastery']}")
print(f"Blind spots: {student_model['blind_spots']}")


## Hattie (2007) 四级 Formative Feedback

> Hattie & Timperley (2007 RER 77(1):81-112) 的 4 级反馈. 本 tutorial 按级给出, 避免仅给 Self 级表扬.

### [TASK] 任务级 - 关于任务本身
- 你的 Thompson MAB 累计转化数字计算正确, 但 regret 曲线未标注 95% 置信带.
- CATE 的 top-3 调节变量识别正确, 但 CausalForestDML 的 `discrete_treatment=True` 参数遗漏, 导致估计偏差.
- Qini 曲线绘制正确, 但拐点判断用肉眼看, 应用 `np.argmax(增量 - 随机基线)`.

### [PROCESS] 过程级 - 关于完成任务的过程/策略
- 你直接调 `econml` API, 但未先做 EDA 检查处理-协变量平衡. 过程上应: 先 `df.groupby('treat')[X].mean()` 检查 RCT 平衡, 再建模.
- 探索-利用权衡的解释, 你只用了代码验证, 未用反事实推理. 过程上应: 先写"若纯利用会怎样? 若纯探索会怎样?"再跑代码.
- 安慰剂检验你只跑 1 次, 应跑 100 次取分布, 这是过程级错误.

### [SELF-REG] 自我调节级 - 关于自我监控/调节
- 你在 D2 失败 2 次后未触发 weak_loop 回退 D1, 而是继续硬刚. 应自我监控: 连续 2 次失败 -> 回退.
- 你跳过了 pre-task 直接进 tutorial, 这违反"没有预先准备=没有学习". 应自我调节: 先写 300 字 pre-task.
- 你未在 student_model.json 更新盲点, 导致下次 tutorial 仍卡同一处. 应: 每次失败后写盲点.

### [FEED-FORWARD] 前馈级 - 关于下一步如何改进
- 下次进 D3 前, 先重做 D2 的安慰剂检验 100 次分布, 把 p 值分布画出来.
- 把"实验成本节省"的定量方法迁移到自选营销场景 (M4), 用真实 DCO 数据重算一遍.
- 在技能3 结业前, 把 Uplift 四类用户映射到你自己工作中的用户分群, 写 1 页迁移报告.

> 故意避免 Self 级 (表扬/批评人格), 因为 Hattie 元分析显示 Self 级反馈效应量最低 (d约0.14), 而 TASK/PROCESS 级效应量最高 (d约0.75-0.90).

## 限频与 Exit (防依赖)

### 限频 (Usage Limit)
- **每单元每天 1 次 tutorial** (限频: 1次/天), 防 LLM 依赖. 参考 Oxford 每周 1 次 + NUS SELENE 自定步调.
- 超过 1 次/天 -> student_model.json 标记 `overuse_flag: true`, 下次 tutorial 自动降级为 "仅 hint, 不 Socratic 追问".
- 跨日累计 5 次 tutorial 仍卡同一 drill -> 触发 1:1 真人 tutorial (转人工).

### Exit Artifact (Tutorial 结束必须提交)
Tutorial 结束前, 学生必须在 student_model.json 写入:
1. **2-3 个盲点** (本轮新发现的, 例如"Beta 后验数学推导""Qini 拐点定义")
2. **推荐复习单元** (跨单元, 例如"回 skill-3-causal/day-2 重学 A/B 基础""回 day-4 重学因果森林")
3. **下次 tutorial 的 pre-task 目标** (具体到 1 个数字, 例如"重算 NSW MAB 节省成本到误差 <5%")

Exit 未提交 -> 本次 tutorial 计 0 分, 须补做.

---

*本 tutorial.ipynb 实现 Oxford tutorial LLM 仿真 (role-engineered persona + Socratic 追问 + 多轮脚手架渐退 + student_model + 限频防依赖) + Hattie 四级反馈 (避开 Self 级). 参考 Vygotsky 共构与 arxiv 2024-2025 Socratic LLM 论文 (2409.05511 / 2507.05795).*
